In [1]:
import os, sys

os.chdir(r"C:\Users\DELL\Desktop\SkillRecommendation-System")
sys.path.insert(0, r"C:\Users\DELL\Desktop\SkillRecommendation-System\projects")

print("Working dir:", os.getcwd())
print("Python path includes projects/:", any("projects" in p for p in sys.path))

Working dir: C:\Users\DELL\Desktop\SkillRecommendation-System
Python path includes projects/: True


In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from data_loader import load_data, build_feature_matrix, get_completed_levels, get_level_pass_rates

TOP_K_NEIGHBOURS = 50
TOP_N_RECOMMENDATIONS = 3
SUCCESS_THRESHOLD = 0.70


class RecommendationEngine:

    def __init__(self, top_k=TOP_K_NEIGHBOURS, top_n=TOP_N_RECOMMENDATIONS, success_threshold=SUCCESS_THRESHOLD):
        self.top_k = top_k
        self.top_n = top_n
        self.success_threshold = success_threshold
        self.df = None
        self.feature_matrix = None
        self.pass_rates = None
        self._similarity_matrix = None

    def fit(self, df):
        self.df = df
        self.feature_matrix = build_feature_matrix(df)
        self.pass_rates = get_level_pass_rates(df)
        matrix = self.feature_matrix.values.astype(float)
        self._similarity_matrix = cosine_similarity(matrix)
        return self

    def recommend(self, student_id):
        if self.feature_matrix is None:
            raise RuntimeError("Call fit() before recommend().")
        if student_id not in self.feature_matrix.index:
            raise ValueError(f"Student {student_id} not found in training data.")

        completed = set(get_completed_levels(self.df, student_id))
        candidate_levels = set(self.feature_matrix.columns.tolist()) - completed
        if not candidate_levels:
            return []

        student_idx = self.feature_matrix.index.get_loc(student_id)
        sim_series = pd.Series(self._similarity_matrix[student_idx], index=self.feature_matrix.index)
        sim_series = sim_series.drop(index=student_id)
        neighbours = sim_series.nlargest(self.top_k)

        neighbour_df = self.df[self.df["student_id"].isin(neighbours.index.tolist())].copy()
        neighbour_df = neighbour_df.merge(neighbours.rename("similarity"), left_on="student_id", right_index=True)
        neighbour_df = neighbour_df[neighbour_df["level_id"].isin(candidate_levels)]
        if neighbour_df.empty:
            return []

        def weighted_pass_rate(group):
            total_weight = group["similarity"].sum()
            return 0.0 if total_weight == 0 else (group["similarity"] * group["passed"]).sum() / total_weight

        level_scores = (
            neighbour_df.groupby("level_id")
            .apply(weighted_pass_rate, include_groups=False)
            .rename("weighted_success")
        )

        recommendations = []
        for level_id, score in level_scores[level_scores >= self.success_threshold].nlargest(self.top_n).items():
            count = neighbour_df[neighbour_df["level_id"] == level_id]["student_id"].nunique()
            recommendations.append({
                "level_id": str(level_id),
                "score": round(float(score), 4),
                "reason": f"{count} similar students attempted '{level_id}' with a {score*100:.1f}% weighted success rate."
            })
        return recommendations

    def evaluate(self, test_student_ids=None):
        if self.df is None:
            raise RuntimeError("Call fit() first.")
        all_ids = self.feature_matrix.index.tolist()
        if test_student_ids is None:
            rng = np.random.default_rng(0)
            test_student_ids = rng.choice(all_ids, size=max(1, len(all_ids) // 10), replace=False).tolist()

        hits, total_recs, students_with_recs = 0, 0, 0
        for sid in test_student_ids:
            recs = self.recommend(sid)
            if not recs:
                continue
            students_with_recs += 1
            passed_levels = set(self.df[(self.df["student_id"] == sid) & (self.df["passed"] == 1)]["level_id"])
            for r in recs:
                total_recs += 1
                if r["level_id"] in passed_levels:
                    hits += 1

        return {
            "precision": round(hits / total_recs if total_recs > 0 else 0.0, 4),
            "coverage": round(students_with_recs / len(test_student_ids) if test_student_ids else 0.0, 4),
            "avg_recommendations": round(total_recs / max(students_with_recs, 1), 2),
            "students_evaluated": len(test_student_ids),
        }


# --- Run ---
df = load_data("datasets/student_progress.csv")
engine = RecommendationEngine()
engine.fit(df)

demo_student = df["student_id"].iloc[0]
completed = get_completed_levels(df, demo_student)
recs = engine.recommend(demo_student)

print(f"\n=== Recommendations for Student {demo_student} ===")
print(f"Completed levels: {completed}")
print(f"\nTop {len(recs)} recommendations:")
for i, r in enumerate(recs, 1):
    print(f"  {i}. Level '{r['level_id']}'  |  score={r['score']:.2%}  |  {r['reason']}")

metrics = engine.evaluate()
print("\n=== Evaluation Metrics (10% sample) ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")


=== Recommendations for Student STU0001 ===
Completed levels: ['algo_greedy', 'algo_sorting', 'ds_dicts', 'math_linear_algebra', 'math_probability', 'py_functions', 'py_loops', 'py_oop', 'py_variables']

Top 3 recommendations:
  1. Level 'ml_clustering'  |  score=100.00%  |  1 similar students attempted 'ml_clustering' with a 100.0% weighted success rate.
  2. Level 'ml_regression'  |  score=100.00%  |  2 similar students attempted 'ml_regression' with a 100.0% weighted success rate.
  3. Level 'ml_classification'  |  score=83.59%  |  6 similar students attempted 'ml_classification' with a 83.6% weighted success rate.

=== Evaluation Metrics (10% sample) ===
  precision: 0.0
  coverage: 0.65
  avg_recommendations: 2.29
  students_evaluated: 100
